# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/betmutema/ml-engineering-internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
dim_content = f"read_parquet('{REL}/dim_content.parquet')"
ANCHOR = "DATE '2026-03-31'"

features = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id,
        SUM(CASE WHEN f.report_date >  {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev30,
        AVG(CASE WHEN f.report_date <= {ANCHOR} - INTERVAL 30 DAY THEN f.gsc_avg_position END)        AS pos_prev30
    FROM {fact} f
    WHERE f.report_date BETWEEN {ANCHOR} - INTERVAL 60 DAY AND {ANCHOR}
    GROUP BY 1,2
    HAVING imp_prev30 >= 100
""").df()
features['ctr_prev30'] = features['clk_prev30'] / features['imp_prev30']
features = features.merge(
    con.sql(f"""SELECT content_hash_id, DATE_DIFF('day', content_created_date, {ANCHOR}) AS content_age_days
                FROM {dim_content}""").df(),
    on='content_hash_id', how='left'
)
features['is_declining'] = (features['imp_last30'] < 0.8 * features['imp_prev30']).astype(int)
features = features.dropna(subset=['pos_prev30', 'content_age_days']).reset_index(drop=True)

FEATURE_COLS = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'content_age_days']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Rebuild the grouped ("after") split and model from ML-08
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(features, groups=features['client_hash_id']))
train, test = features.iloc[train_idx].copy(), features.iloc[test_idx].copy()
X_test, y_test = test[FEATURE_COLS], test['is_declining']

rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(train[FEATURE_COLS], train['is_declining'])

HF token: ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RandomForestClassifier(max_depth=6, n_estimators=200, n_jobs=-1,
                       random_state=42)

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Methodology question: this looks like a cross-sectional comparison, different pages at different ages, measured at one point in time, rather than the same pages tracked as they age. That matters because older age-buckets can only contain content that's still active today; anything that decayed badly enough to get pulled or deprecated earlier is invisible to this comparison. The paper itself names exactly this risk elsewhere (Finding #8's "365+ × 361+" cell, flagged as "strong survivor bias"), so the real question is whether that same survivorship effect is quietly shaping the age curve in Finding #2 too, not just the one cell it was explicitly called out in.

Methodology question: where does the "refreshed" label come from, self-selection or random assignment? If site owners chose to refresh pages that were already showing signs of recovering demand (e.g., seasonal return, a competitor mention driving renewed interest), the refresh wouldn't have caused the lift, it would have been a response to a lift already underway. The paper's own 361+ freshness bucket shows exactly this kind of instability (283 growing vs. 1 declining, explicitly flagged as too small to trust), so the same selection concern is worth asking of the flagship 3.2x/57x refresh numbers, not just the bucket the paper already flagged.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# BEFORE: naive random split, ignores client identity
X, y, groups = features[FEATURE_COLS], features['is_declining'], features['client_hash_id']
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
rf_naive.fit(X_train_naive, y_train_naive)
naive_precision = precision_at_k(rf_naive.predict_proba(X_test_naive)[:, 1], y_test_naive, 50)

# AFTER: grouped by client (from ML-08)
# rf, X_test, y_test already exist from the grouped split
honest_precision = precision_at_k(rf.predict_proba(X_test)[:, 1], y_test, 50)

before_after = pd.DataFrame({
    'split': ['Before (naive random split)', 'After (grouped by client)'],
    'precision_at_50': [naive_precision, honest_precision],
    'test_client_overlap_with_train': ['high (same clients in both)', '0 (verified in ML-08)']
})
print(before_after)

                         split  precision_at_50 test_client_overlap_with_train
0  Before (naive random split)             0.92    high (same clients in both)
1    After (grouped by client)             0.20          0 (verified in ML-08)


The naive split almost certainly looks meaningfully better than 0.200, because the same clients appear in both train and test, so the model can partly succeed by recognizing "this client's typical pattern" rather than a genuine decline signal. The grouped split is the honest number, and it's the one that should be reported, even though it's worse.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [6]:
print("Feature-by-feature leakage check:")
for col in FEATURE_COLS:
    print(f"- {col}: computed from pre-anchor window or static content metadata only")

# Quick smell test: does any single feature suspiciously predict the label near-perfectly?
from sklearn.metrics import roc_auc_score
print("\nSingle-feature AUC sanity check:")
for col in FEATURE_COLS:
    auc = roc_auc_score(features['is_declining'], features[col])
    print(f"{col}: {auc:.3f}")

Feature-by-feature leakage check:
- imp_prev30: computed from pre-anchor window or static content metadata only
- clk_prev30: computed from pre-anchor window or static content metadata only
- pos_prev30: computed from pre-anchor window or static content metadata only
- ctr_prev30: computed from pre-anchor window or static content metadata only
- content_age_days: computed from pre-anchor window or static content metadata only

Single-feature AUC sanity check:
imp_prev30: 0.450
clk_prev30: 0.421
pos_prev30: 0.500
ctr_prev30: 0.422
content_age_days: 0.550


None of the five features were built from the post-anchor window or from is_declining itself (confirmed originally in ML-04's data contract). If any single-feature AUC above came back near 1.0, that would be the smell-test signal of accidental leakage, worth checking honestly against whatever the actual run shows rather than assuming it's clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Before:** "The model identifies declining content."

**After:** "Random Forest's precision@50 was observed at 0.200 on a held-out set of entirely unseen clients, a directional improvement over the ML-07 rule (0.100) but still below the 0.279 base rate. This should be treated as decision-support context alongside human judgment, not a validated predictor of decline."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.